# n-Grams and Grammars
__MATH 3480__ - Dr. Michael Olson

Reading:
* Grus, Chapter 21
  * Sections: n-Gram Language Models, Grammars

## N‑grams and Grammars in Natural Language Processing

Natural language is discrete,variable‑length,and highly structured. Two of the most important modeling ideas in classical Natural Language Processing(NLP)are**n‑gram language models**and**grammars**.  

They represent two complementary views of language:
- **N‑grams** model language *statistically* as sequences of tokens. 
- **Grammars** model language *structurally* as recursive compositional rules.

Both ideas predate modern neural models,but they remain foundational for understanding how language models work and why learning language is difficult.

### Tokens and Sequences

We begin by representing text as a sequence of discrete tokens. A *token* may be a word, subword, or character.

A sentence with tokens $\{w_1,w_2,\dots,w_T\}$ is treated as a realization of a random sequence.

The goal of a **language model** is to assign a probability to the entire sequence:

$$P(w_1,w_2,\dots,w_T)$$

By the chain rule of probability,

$$P(w_1,\dots,w_T) = \prod_{t=1}^T P(w_t \mid w_1,\dots,w_{t-1})$$

The key modeling challenge is how to represent the conditional distributions on the right‑hand side.

-----

## N‑gram Language Models

### The Markov Assumption

An **n‑gram model** approximates the full conditional distribution by assuming that only the previous $n-1$ tokens matter:

$$P(w_t \mid w_1,\dots,w_{t-1})\approx P(w_t \mid w_{t-(n-1)},\dots,w_{t-1})$$

This is a fixed‑order **Markov assumption**.

Common cases:
- **Unigram** ($n=1$): no context
- **Bigram** ($n=2$): one previous token
- **Trigram** ($n=3$): two previous tokens  

With this assumption,

$$P(w_1,\dots,w_T)\approx \prod_{t=1}^T P(w_t \mid w_{t-(n-1)},\dots,w_{t-1})$$

-----

### Maximum Likelihood Estimation

N‑gram probabilities are typically estimated by **relative frequency**:

$$
P(w_t \mid w_{t-(n-1)},\dots,w_{t-1})=\frac{
\text{count}(w_{t-(n-1)},\dots,w_t)}{
\text{count}(w_{t-(n-1)},\dots,w_{t-1})}
$$

This is the maximum likelihood estimator under the multinomial model.

To handle sentence boundaries,special tokens such as `<s>` and `</s>` are often added.

-----
## bigram

In [1]:
def fix_unicode(text: str) -> str:
    return text.replace(u"\u2019", "'")

In [2]:
import re
from numpy import random

file = open('Data/WhatIsDataScience.txt')
content = file.read()
content = content.split('\n')

content

UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 176: character maps to <undefined>

In [ ]:
regex = r"[\w']+|[\.]"                       # matches a word or a period

document = []

for paragraph in content:
     if not paragraph == '':
        words = re.findall(regex, fix_unicode(paragraph))
        document.extend(words)

document

In [ ]:
from collections import defaultdict

transitions = defaultdict(list)
for prev, current in zip(document, document[1:]):
    transitions[prev].append(current)

In [ ]:
def generate_using_bigrams() -> str:
    current = "."   # this means the next word will start a sentence
    result = []
    while True:
        next_word_candidates = transitions[current]    # bigrams (current, _)
        current = random.choice(next_word_candidates)  # choose one at random
        result.append(current)                         # append it to results
        if current == ".": return " ".join(result)     # if "." we're done

In [ ]:
generate_using_bigrams()

## trigram

In [ ]:
trigram_transitions = defaultdict(list)
starts = []

for prev, current, next in zip(document, document[1:], document[2:]):

    if prev == ".":              # if the previous "word" was a period
        starts.append(current)   # then this is a start word

    trigram_transitions[(prev, current)].append(next)

In [ ]:
def generate_using_trigrams() -> str:
    current = random.choice(starts)   # choose a random starting word
    prev = "."                        # and precede it with a '.'
    result = [current]
    while True:
        next_word_candidates = trigram_transitions[(prev, current)]
        next_word = random.choice(next_word_candidates)

        prev, current = current, next_word
        result.append(current)

        if current == ".":
            return " ".join(result)

In [ ]:
generate_using_trigrams()

## n-gram

In [ ]:
from nltk import ngrams

n = 6
sixgrams = ngrams(document, n)

for grams in sixgrams:
  print(grams)

-----
## Grammars

__Grammars__ are based on the structure of sentences instead of the probability of word sequences. We identify words as nouns, verbs, adjectives, etc., then randomly select words to fit the word types to fit a given word structure.

![Grammar Examples](https://raw.githubusercontent.com/drolsonmi/math3480/refs/heads/main/Notes/Images/NLP_GrammarExamples.jpg)

To create sentences using grammars,
- Randomly create a sentence structure
- Randomly select words to fit that structure

Best practice would be to combine the n-grams and grammar methods.

In [ ]:
from typing import List, Dict

# Type alias to refer to grammars later
Grammar = Dict[str, List[str]]

grammar = {
    "_S"  : ["_NP _VP"],
    "_NP" : ["_N",
             "_A _NP _P _A _N"],
    "_VP" : ["_V",
             "_V _NP"],
    "_N"  : ["data science", "Python", "regression"],
    "_A"  : ["big", "linear", "logistic"],
    "_P"  : ["about", "near"],
    "_V"  : ["learns", "trains", "tests", "is"]
}

def is_terminal(token: str) -> bool:
    return token[0] != "_"

In [ ]:
def expand(grammar: Grammar, tokens: List[str]) -> List[str]:
    for i, token in enumerate(tokens):
        # If this is a terminal token, skip it.
        if is_terminal(token): continue

        # Otherwise, it's a nonterminal token,
        # so we need to choose a replacement at random.
        replacement = random.choice(grammar[token])

        if is_terminal(replacement):
            tokens[i] = replacement
        else:
            # Replacement could be, e.g., "_NP _VP", so we need to
            # split it on spaces and splice it in.
            tokens = tokens[:i] + replacement.split() + tokens[(i+1):]

        # Now call expand on the new list of tokens.
        return expand(grammar, tokens)

    # If we get here, we had all terminals and are done.
    return tokens

In [ ]:
def generate_sentence(grammar: Grammar) -> List[str]:
    return expand(grammar, ["_S"])

In [ ]:
generate_sentence(grammar)